# WM-811K Wafer Map — Training Pipeline

Load the dataset, filter out unknown failure types, stratified 60/20/20 split, and build PyTorch `Dataset` / `DataLoader`.

In [22]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import zoom
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Load the data

In [23]:
import pickle
import sys

# Patch old pandas module paths for compatibility
# The pickle was saved with pandas < 0.20 which used pandas.indexes.*
_compat_map = {
    'pandas.indexes':         'pandas.core.indexes',
    'pandas.indexes.base':    'pandas.core.indexes.base',
    'pandas.indexes.numeric': 'pandas.core.indexes.base',
    'pandas.indexes.range':   'pandas.core.indexes.range_',
    'pandas.indexes.multi':   'pandas.core.indexes.multi',
    'pandas.indexes.frozen':  'pandas.core.indexes.frozen',
}

for old, new in _compat_map.items():
    try:
        sys.modules[old] = __import__(new, fromlist=[''])
    except ImportError:
        pass

class _CompatUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        for old, new in _compat_map.items():
            if module.startswith(old):
                module = module.replace(old, new, 1)
                break
        return super().find_class(module, name)

with open("./Data/LSWMD.pkl", "rb") as f:
    try:
        df = pickle.load(f, encoding='latin1')
    except (ModuleNotFoundError, ImportError):
        f.seek(0)
        df = _CompatUnpickler(f)
        df.encoding = 'latin1'
        df = df.load()

print(f"Shape: {df.shape}")
df.head()

/var/folders/k2/4b0bgh394psfjhjl_xtj93gh000585/T/ipykernel_41113/2994757470.py:31: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  df = pickle.load(f, encoding='latin1')


Shape: (811457, 6)


,waferMap,dieSize,lotName,waferIndex,trianTestLabel,failureType
0,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,1.0,[[Training]],[[none]]
1,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,2.0,[[Training]],[[none]]
2,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,3.0,[[Training]],[[none]]
3,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,4.0,[[Training]],[[none]]
4,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,5.0,[[Training]],[[none]]


## 2. Extract labels and filter out unknown failure types

The `failureType` column is stored as a nested array. Rows with an empty array (no label) or `'unknown'` are dropped — keeping only the 9 named defect classes plus `none`.

In [24]:
def extract_label(x):
    if isinstance(x, (list, np.ndarray)):
        flat = np.array(x).flatten()
        if len(flat) > 0:
            return str(flat[0])
    if isinstance(x, str):
        return x
    return "unknown"

df['failure_label'] = df['failureType'].apply(extract_label)

# Drop unlabeled rows
df_labeled = df[~df['failure_label'].isin(['unknown', ''])].reset_index(drop=True)

print(f"Before filter: {len(df):,}")
print(f"After filter:  {len(df_labeled):,}")
print(f"\nClass counts:")
print(df_labeled['failure_label'].value_counts())

Before filter: 811,457
After filter:  172,950

Class counts:
failure_label
none         147431
Edge-Ring      9680
Edge-Loc       5189
Center         4294
Loc            3593
Scratch        1193
Random          866
Donut           555
Near-full       149
Name: count, dtype: int64


## 3. Train / val / test split

Test set comes from `trianTestLabel == 'Test'` (the dataset's pre-assigned holdout). The `Training` rows are split 90 / 10 (stratified by class) into train and validation.

In [25]:
classes = sorted(df_labeled['failure_label'].unique().tolist())
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
NUM_CLASSES = len(classes)
print(f"{NUM_CLASSES} classes: {classes}")


df_labeled['label_idx'] = df_labeled['failure_label'].map(class_to_idx)


# Use the dataset's pre-assigned Train/Test split from `trianTestLabel`.
def _extract_split(x):
    if isinstance(x, (list, np.ndarray)):
        flat = np.array(x).flatten()
        if len(flat) > 0:
            return str(flat[0])
    return str(x) if isinstance(x, str) else ''


df_labeled['split_label'] = df_labeled['trianTestLabel'].apply(_extract_split)
df_labeled = df_labeled[df_labeled['split_label'].isin(['Training', 'Test'])].reset_index(drop=True)


y = df_labeled['label_idx'].values
idx_test     = np.where(df_labeled['split_label'].values == 'Test')[0]
idx_trainval = np.where(df_labeled['split_label'].values == 'Training')[0]


# Split the Training rows 90/10 into train/val (stratified by class).
idx_train, idx_val = train_test_split(
    idx_trainval, test_size=0.10,
    stratify=y[idx_trainval], random_state=SEED,
)


total = len(df_labeled)
print(f"\nTrain: {len(idx_train):,}  ({len(idx_train)/total:.1%})")
print(f"Val:   {len(idx_val):,}  ({len(idx_val)/total:.1%})")
print(f"Test:  {len(idx_test):,}  ({len(idx_test)/total:.1%})")


# Verify class proportions match across splits
prop_table = pd.DataFrame({
    'all':   df_labeled['failure_label'].value_counts(normalize=True),
    'train': df_labeled.iloc[idx_train]['failure_label'].value_counts(normalize=True),
    'val':   df_labeled.iloc[idx_val]['failure_label'].value_counts(normalize=True),
    'test':  df_labeled.iloc[idx_test]['failure_label'].value_counts(normalize=True),
}).fillna(0).round(4)
print("\nClass proportions per split:")
prop_table



9 classes: ['Center', 'Donut', 'Edge-Loc', 'Edge-Ring', 'Loc', 'Near-full', 'Random', 'Scratch', 'none']

Train: 48,919  (28.3%)
Val:   5,436  (3.1%)
Test:  118,595  (68.6%)

Class proportions per split:


,all,train,val,test
failure_label,,,,
Center,0.0248,0.0637,0.0636,0.0070
Donut,0.0032,0.0075,0.0075,0.0012
Edge-Loc,0.0300,0.0445,0.0445,0.0234
Edge-Ring,0.0560,0.1574,0.1575,0.0095
Loc,0.0208,0.0298,0.0298,0.0166
Near-full,0.0009,0.0010,0.0009,0.0008
Random,0.0050,0.0112,0.0112,0.0022
Scratch,0.0069,0.0092,0.0092,0.0058
none,0.8524,0.6757,0.6757,0.9334


## 4. PyTorch `Dataset`

Wafer maps come in many sizes (heights/widths range widely) so each map is resized to a fixed `IMG_SIZE × IMG_SIZE` with nearest-neighbor (preserves the discrete 0/1/2 codes). The result is returned as a `(1, H, W)` float tensor in `[0, 1]` plus the integer class label.

Training instances also get random horizontal + vertical flips (each with p = 0.5). Wafer-map defects don't have a canonical orientation, so flipping is label-preserving — it effectively quadruples the variety the model sees per class, which is especially useful for the rare classes the weighted sampler keeps drawing. Validation and test stay deterministic.

In [26]:
IMG_SIZE = 64

class WaferMapDataset(Dataset):
    def __init__(self, df, indices, img_size=IMG_SIZE, augment: bool = False):
        self.maps    = df['waferMap'].values[indices]
        self.labels  = df['label_idx'].values[indices].astype(np.int64)
        self.img_size = img_size
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def _resize(self, wmap):
        wmap = np.asarray(wmap)
        if wmap.ndim < 2 or wmap.shape[0] == 0 or wmap.shape[1] == 0:
            return np.zeros((self.img_size, self.img_size), dtype=np.float32)
        h, w = wmap.shape
        return zoom(wmap, (self.img_size / h, self.img_size / w), order=0)

    def __getitem__(self, i):
        wmap = self._resize(self.maps[i]).astype(np.float32) / 2.0  # 0/1/2 -> [0, 0.5, 1]
        if self.augment:
            # Independent p=0.5 horizontal and vertical flips. Wafer defects have no
            # canonical orientation so flips are label-preserving.
            if np.random.rand() < 0.5:
                wmap = np.ascontiguousarray(wmap[:, ::-1])
            if np.random.rand() < 0.5:
                wmap = np.ascontiguousarray(wmap[::-1, :])
        x = torch.from_numpy(wmap).unsqueeze(0)                     # (1, H, W)
        y = torch.tensor(self.labels[i], dtype=torch.long)
        return x, y

train_ds = WaferMapDataset(df_labeled, idx_train, augment=True)
val_ds   = WaferMapDataset(df_labeled, idx_val,   augment=False)
test_ds  = WaferMapDataset(df_labeled, idx_test,  augment=False)

print(f"train_ds: {len(train_ds):,}  (augmented)")
print(f"val_ds:   {len(val_ds):,}")
print(f"test_ds:  {len(test_ds):,}")

x0, y0 = train_ds[0]
print(f"\nsample x: shape={tuple(x0.shape)}, dtype={x0.dtype}, min={x0.min():.2f}, max={x0.max():.2f}")
print(f"sample y: {y0.item()} ({idx_to_class[y0.item()]})")

train_ds: 48,919  (augmented)
val_ds:   5,436
test_ds:  118,595

sample x: shape=(1, 64, 64), dtype=torch.float32, min=0.00, max=1.00
sample y: 8 (none)


## 5. `DataLoader`s

In [27]:
from torch.utils.data import WeightedRandomSampler

BATCH_SIZE  = 64
NUM_WORKERS = 0

# Per-sample weights: 1 / count[class]. Rare classes get higher draw probability,
# so each batch ends up with ~uniform class composition. Replacement=True is required
# (otherwise rare classes run out and the distribution skews back toward the majority).
train_labels = df_labeled['label_idx'].values[idx_train]
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_labels]

train_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_labels),  # one "epoch" = same #steps as natural dataset
    replacement=True,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler,
                          num_workers=NUM_WORKERS, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS)

# --- class counts BEFORE / AFTER augmentation ---
# "Before" = raw training set (pre-sampler, no flips).
# "After"  = one epoch as the model actually sees it: WeightedRandomSampler drawing
#            with replacement, where each draw also gets an independent random
#            hflip+vflip from WaferMapDataset(augment=True).
#            We don't load images here — we just iterate sampler indices and look up
#            their labels. Flipping doesn't change a sample's class, so this is the
#            true post-augmentation class distribution.
g = torch.Generator().manual_seed(SEED)
sim_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_labels),
    replacement=True,
    generator=g,
)
sampled_idx = np.fromiter(iter(sim_sampler), dtype=np.int64, count=len(train_labels))
after_counts = np.bincount(train_labels[sampled_idx], minlength=NUM_CLASSES)

print(f"{'class':12s} {'before':>10s} {'after':>10s} {'before %':>10s} {'after %':>10s}")
print('-' * 56)
tot_b, tot_a = class_counts.sum(), after_counts.sum()
for i in range(NUM_CLASSES):
    print(f"{idx_to_class[i]:12s} "
          f"{class_counts[i]:>10,d} {after_counts[i]:>10,d} "
          f"{class_counts[i]/tot_b:>9.2%} {after_counts[i]/tot_a:>9.2%}")
print('-' * 56)
print(f"{'total':12s} {tot_b:>10,d} {tot_a:>10,d}")

# DataLoader sanity check on one batch
xb, yb = next(iter(train_loader))
print(f"\nbatch x: {tuple(xb.shape)}  dtype={xb.dtype}")
print(f"batch y: {tuple(yb.shape)}  dtype={yb.dtype}")

class            before      after   before %    after %
--------------------------------------------------------
Center            3,116      5,505     6.37%    11.25%
Donut               368      5,485     0.75%    11.21%
Edge-Loc          2,175      5,410     4.45%    11.06%
Edge-Ring         7,698      5,441    15.74%    11.12%
Loc               1,458      5,365     2.98%    10.97%
Near-full            49      5,494     0.10%    11.23%
Random              548      5,411     1.12%    11.06%
Scratch             450      5,457     0.92%    11.16%
none             33,057      5,351    67.57%    10.94%
--------------------------------------------------------
total            48,919     48,919

batch x: (64, 1, 64, 64)  dtype=torch.float32
batch y: (64,)  dtype=torch.int64


## 6. Model 1 — Simple CNN

In [28]:
import torch
import torch.nn as nn


class Model2(nn.Module):
    """Simple CNN with BatchNorm to fight overfitting.

    Normalization additions vs. the original:
      - BatchNorm2d after each conv (before ReLU). Stabilizes activations and
        also acts as a regularizer because batch statistics inject noise.
      - BatchNorm1d after the first FC layer.
      - Light spatial dropout (Dropout2d) in the conv stack on top of the existing
        FC dropout, so regularization isn't only at the very end.
    """
    def __init__(self, num_classes: int = 9, in_channels: int = 1):
        super().__init__()

        # Convolutional part
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=6, padding=1)
        self.bn1   = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU(inplace=True)
        self.pool1 = nn.MaxPool2d(2)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=6, padding=1)
        self.bn2   = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU(inplace=True)
        self.pool2 = nn.MaxPool2d(2)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=6, padding=1)
        self.bn3   = nn.BatchNorm2d(128)
        self.relu3 = nn.ReLU(inplace=True)
        self.pool3 = nn.MaxPool2d(2)

        self.conv_dropout = nn.Dropout2d(p=0.1)

        # Dry-run to infer flattened size
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, 64, 64)
            dummy = self.pool3(self.relu3(self.bn3(self.conv3(
                     self.pool2(self.relu2(self.bn2(self.conv2(
                     self.pool1(self.relu1(self.bn1(self.conv1(dummy)))))))))))) 
            flat_size = dummy.numel()

        # Fully connected part
        self.flatten = nn.Flatten()
        self.fc1     = nn.Linear(flat_size, 256)
        self.bn_fc1  = nn.BatchNorm1d(256)
        self.relu4   = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2     = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = self.pool3(self.relu3(self.bn3(self.conv3(x))))
        x = self.conv_dropout(x)

        x = self.flatten(x)
        x = self.relu4(self.bn_fc1(self.fc1(x)))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"device: {device}")

# Param count via a throwaway instance — does NOT bind `model1`, so re-running this
# cell never resets a trained model. The actual `model1` is built in the training cell.
_probe = Model2(num_classes=NUM_CLASSES)
print(f"Model1 parameters: {sum(p.numel() for p in _probe.parameters()):,}")
del _probe

device: mps
Model1 parameters: 1,192,745


## 7. Train Model 2

In [29]:
import time

EPOCHS = 50
LR     = 1e-3


model2 = Model2(num_classes=NUM_CLASSES).to(device)

# Forward-pass sanity check
with torch.no_grad():
    out = model2(xb.to(device))
print(f"forward output: {tuple(out.shape)}  (expected ({BATCH_SIZE}, {NUM_CLASSES}))")

optimizer = torch.optim.Adam(model2.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()


def run_epoch(model, loader, train: bool):
    model.train(train)
    total_loss = 0.0
    total_correct = 0
    total_seen = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss    += loss.item() * yb.size(0)
            total_correct += (logits.argmax(1) == yb).sum().item()
            total_seen    += yb.size(0)
    return total_loss / total_seen, total_correct / total_seen


history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(model2, train_loader, train=True)
    va_loss, va_acc = run_epoch(model2, val_loader,   train=False)
    history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
    history['val_loss'].append(va_loss);   history['val_acc'].append(va_acc)
    print(f"epoch {epoch:2d} | "
          f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
          f"val loss {va_loss:.4f} acc {va_acc:.4f} | "
          f"{time.time()-t0:.1f}s")


forward output: (64, 9)  (expected (64, 9))
epoch  1 | train loss 0.4634 acc 0.8401 | val loss 1.0952 acc 0.5408 | 15.9s
epoch  2 | train loss 0.2546 acc 0.9115 | val loss 1.7894 acc 0.4284 | 15.4s
epoch  3 | train loss 0.1777 acc 0.9388 | val loss 0.2089 acc 0.9218 | 15.5s
epoch  4 | train loss 0.1374 acc 0.9523 | val loss 0.4318 acc 0.8521 | 15.5s
epoch  5 | train loss 0.1135 acc 0.9605 | val loss 0.1071 acc 0.9650 | 15.5s
epoch  6 | train loss 0.0941 acc 0.9675 | val loss 0.6111 acc 0.8469 | 15.5s
epoch  7 | train loss 0.0769 acc 0.9731 | val loss 0.0738 acc 0.9737 | 15.5s
epoch  8 | train loss 0.0716 acc 0.9753 | val loss 0.0616 acc 0.9823 | 15.5s
epoch  9 | train loss 0.0630 acc 0.9784 | val loss 0.0666 acc 0.9796 | 15.5s
epoch 10 | train loss 0.0563 acc 0.9810 | val loss 0.0755 acc 0.9744 | 15.5s
epoch 11 | train loss 0.0536 acc 0.9821 | val loss 0.1948 acc 0.9343 | 15.5s
epoch 12 | train loss 0.0468 acc 0.9841 | val loss 0.0503 acc 0.9842 | 15.5s
epoch 13 | train loss 0.0428 acc

## 8. Evaluate Model 2 on the test set

In [30]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, cohen_kappa_score,
)


@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        ps.append(model(xb).argmax(1).cpu().numpy())
        ys.append(yb.numpy())
    return np.concatenate(ys), np.concatenate(ps)


y_true, y_pred = collect_predictions(model2, test_loader)
target_names = [idx_to_class[i] for i in range(NUM_CLASSES)]

print(f"Test samples: {len(y_true):,}")
print(f"\nAccuracy:           {accuracy_score(y_true, y_pred):.4f}")
print(f"Balanced accuracy:  {balanced_accuracy_score(y_true, y_pred):.4f}  (mean per-class recall)")
print(f"Cohen's kappa:      {cohen_kappa_score(y_true, y_pred):.4f}")
for avg in ('macro', 'weighted'):
    p = precision_score(y_true, y_pred, average=avg, zero_division=0)
    r = recall_score(   y_true, y_pred, average=avg, zero_division=0)
    f = f1_score(       y_true, y_pred, average=avg, zero_division=0)
    print(f"{avg:>8s}: precision {p:.4f}  recall {r:.4f}  f1 {f:.4f}")

print("\nPer-class report:")
print(classification_report(y_true, y_pred, target_names=target_names,
                            digits=4, zero_division=0))

# Confusion matrix as a labeled DataFrame (rows = true, columns = predicted)
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)
print("Confusion matrix (rows=true, cols=predicted):")
cm_df

Test samples: 118,595

Accuracy:           0.9217
Balanced accuracy:  0.6828  (mean per-class recall)
Cohen's kappa:      0.5166
   macro: precision 0.6257  recall 0.6828  f1 0.6314
weighted: precision 0.9490  recall 0.9217  f1 0.9331

Per-class report:
              precision    recall  f1-score   support

      Center     0.4117    0.7368    0.5282       832
       Donut     0.7350    0.5890    0.6540       146
    Edge-Loc     0.4177    0.6919    0.5209      2772
   Edge-Ring     0.8391    0.6066    0.7041      1126
         Loc     0.4662    0.5352    0.4983      1973
   Near-full     0.9551    0.8947    0.9239        95
      Random     0.7121    0.7121    0.7121       257
     Scratch     0.1123    0.4358    0.1786       693
        none     0.9821    0.9429    0.9621    110701

    accuracy                         0.9217    118595
   macro avg     0.6257    0.6828    0.6314    118595
weighted avg     0.9490    0.9217    0.9331    118595

Confusion matrix (rows=true, cols=predict

,Center,Donut,Edge-Loc,Edge-Ring,Loc,Near-full,Random,Scratch,none
Center,613,9,5,1,13,0,0,6,185
Donut,9,86,1,0,24,0,5,0,21
Edge-Loc,17,1,1918,52,148,1,24,21,590
Edge-Ring,2,1,139,683,2,0,8,4,287
Loc,95,13,200,0,1056,1,10,57,541
Near-full,0,0,3,0,0,85,7,0,0
Random,12,2,19,0,17,2,183,0,22
Scratch,5,3,27,1,103,0,0,302,252
none,736,2,2280,77,902,0,20,2299,104385
